In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import RobustScaler
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

df = pd.read_csv(
    "../datasets/engineered/D5_feature_engineered.csv"
)

print("D5 Dataset Loaded")
print(df.shape)


D5 Dataset Loaded
(119398, 49)


In [2]:
df = df.drop(
    columns=[
        "last_updated",
        "sunrise",
        "sunset",
        "moonrise",
        "moonset"
    ]
)
print(df.shape)


(119398, 44)


In [3]:
categorical_columns = df.select_dtypes(
    include=["object", "string"]
).columns
print(categorical_columns)



Index(['location_name', 'region', 'timezone', 'condition_text', 'wind_direction', 'moon_phase', 'day_period', 'season'], dtype='str')


In [5]:
encoders = {}
for col in categorical_columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le


In [6]:
print(df.shape)
print(
    df.select_dtypes(
        include=["object"]
    ).columns
)


(119398, 44)
Index([], dtype='str')


In [7]:
import joblib
import os
os.makedirs(
    "../artifacts",
    exist_ok=True
)
joblib.dump(
    encoders,
    "../artifacts/label_encoders.pkl"
)
print("Encoders Saved")


Encoders Saved


In [8]:
df.to_csv(
    "../datasets/processed/D6_label_encoded.csv",
    index=False
)

print("Label Encoded Dataset Saved")

df = df.drop(
    columns=[
        "last_updated_epoch",
        "feels_like_fahrenheit",
        "location_name"
    ]
)

print(df.shape)


Label Encoded Dataset Saved
(119398, 41)


In [9]:
scaler = RobustScaler()
joblib.dump(
    scaler,
    "../artifacts/robust_scaler.pkl"
)
print("Scaler Saved")


Scaler Saved


In [10]:
scaled_df = pd.DataFrame(
    scaler.fit_transform(df),
    columns=df.columns
)
scaled_df.to_csv(
    "../datasets/processed/D6_scaled.csv",
    index=False
)
print("Scaled Dataset Saved")


Scaled Dataset Saved


In [11]:
kmeans_df = pd.get_dummies(
    df,
    columns=[
        "region",
        "timezone",
        "condition_text",
        "wind_direction",
        "moon_phase",
        "day_period",
        "season"
    ],
    drop_first=True
)

print(kmeans_df.shape)
kmeans_df.to_csv(
    "../datasets/processed/D6_kmeans_ready.csv",
    index=False
)
print("KMeans Dataset Saved")


(119398, 144)
KMeans Dataset Saved


In [12]:
print("Original Shape:", df.shape)
print("Scaled Shape:", scaled_df.shape)
print(
    scaled_df.head()
)


Original Shape: (119398, 41)
Scaled Shape: (119398, 41)
   region  latitude  longitude  timezone  temperature_celsius  condition_text  wind_kph  wind_degree  wind_direction  pressure_mb  precip_mm  humidity  cloud  feels_like_celsius  visibility_km  \
0     0.0  0.095385  -0.123883       0.0             0.759036          1.8750  2.245902     0.720183        1.285714    -0.833333        0.0  0.097561  0.450            0.885417            0.0   
1     0.0 -0.095385  -0.113665       0.0             0.759036          2.8125  1.426230     0.747706        1.285714    -0.833333        0.0  0.170732  0.275            0.916667            0.0   
2     0.0 -0.289231   0.030651       0.0             0.614458          1.8750  1.901639     0.885321        0.285714    -0.666667        0.0  0.170732  1.075            0.729167            0.0   
3     0.0 -0.321538  -0.097063       0.0             0.530120          0.0625  1.655738     0.793578        1.285714    -0.666667        0.0  0.317073  1.425   

In [13]:
os.makedirs(
   "../reports",
   exist_ok=True
)

with open("../reports/preprocessing_report.txt", "w", encoding="utf-8") as f:

    f.write("""

CLIMATEGUARD AI PREPROCESSING REPORT

D1 Dataset Loading
✓ Completed

D2 Data Quality Verification
✓ Missing Values = 0
✓ Duplicate Records = 0

D3 Feature Removal
✓ Removed 7 redundant features

D4 Datetime Processing
✓ year
✓ month
✓ day
✓ hour
✓ weekday
✓ day_length_minutes
Datetime Cleanup
Moonrise/Moonset Removal

D5 Feature Engineering
✓ temperature_gap
✓ pm_difference
✓ pollution_intensity
✓ wind_humidity_interaction
✓ humidity_cloud_interaction
✓ heatwave_index
✓ day_period
✓ season

D6 Encoding and Scaling
✓ Label Encoding
✓ Robust Scaling

Outlier Handling
✓ Preserved
✓ RobustScaler applied

Feature Selection
✓ Deferred to modelling phase

Generated Datasets
✓ D3_feature_removed.csv
✓ D4_datetime_processed.csv
✓ D5_feature_engineered.csv
✓ D6_label_encoded.csv
✓ D6_scaled.csv
✓ D6_kmeans_ready.csv
Dataset ready for:
1. KMeans Clustering
2. Isolation Forest
3. Rainfall Prediction
4. Heatwave Prediction
5. Climate Risk Engine

Generated Artifacts
✓ label_encoders.pkl
✓ robust_scaler.pkl
""")

print("Preprocessing Report Saved")


Preprocessing Report Saved
